# arc3-momentum-haiku-smoke

CPU + internet smoke test of the momentum time policy, NOT a submission. The anim notebook with the vLLM
boot removed and Claude Haiku as the analyzer, under the served model's harness settings. 6 games on 3 slots.


In [ ]:
# [calamitychasm] ADDED FOR THIS FORK -- diagnostic only, changes no behaviour.
# The upstream README warns that a manual copy must select the RTX PRO 6000 by hand.
# We push via the API with machine_shape=NvidiaRtxPro6000, which IS honoured (verified
# in experiments/stage7_duck_nvfp4.md), but a wrong card would waste the whole run, so
# print what we actually got before anything expensive happens.
import os, shutil, subprocess

print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap,driver_version",
     "--format=csv,noheader"],
    capture_output=True, text=True).stdout.strip() or "nvidia-smi unavailable")
try:
    _mem_kb = int(next(l.split()[1] for l in open("/proc/meminfo") if l.startswith("MemTotal")))
    _ram = f"{_mem_kb / 1048576:.1f}"
except Exception:
    _ram = "?"
print(f"HW_PROBE host_ram_gib={_ram}  cpu_count={os.cpu_count()}  "
      f"free_disk_gib={shutil.disk_usage('/kaggle/working').free / 2**30:.1f}  "
      f"rerun={os.getenv('KAGGLE_IS_COMPETITION_RERUN')!r}")


## 1. Environment and submission mode

Detect whether this is a real competition rerun (which minimises diagnostics), set the
framework's environment flags, and put the CUDA libraries on the linker path.

In [ ]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# Skip periodic JSON/HTML diagnostics and per-frame logging for every run.
# thui-animfast: full diagnostics on an interactive public run (usage/events/transcript sidecars); minimal in a rerun.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"

# Apply the measured vLLM winner before any serving setup command runs.
PUBLIC25_VLLM_PROFILE_NAME = 'kv5-bf16-mtp3-c8-cg32'
PUBLIC25_VLLM_PROFILE_ENV = {
    "TAAF_VLLM_ENABLE_PREFIX_CACHING": "0",
    "TAAF_VLLM_KV_CACHE_DTYPE": "auto",
    "TAAF_VLLM_KV_CACHE_MEMORY_BYTES": "5368709120",
    "TAAF_VLLM_MAX_CUDAGRAPH_CAPTURE_SIZE": "32",
    "TAAF_VLLM_MAX_NUM_BATCHED_TOKENS": "8192",
    "TAAF_VLLM_MAX_NUM_SEQS": "8",
    "TAAF_VLLM_MTP_TOKENS": "3",
    "TAAF_VLLM_OMP_THREADS": "1"
}
for key, value in PUBLIC25_VLLM_PROFILE_ENV.items():
    os.environ[key] = value
print(
    f'PUBLIC25_VLLM_PROFILE name={PUBLIC25_VLLM_PROFILE_NAME} '
    f'env={json.dumps(PUBLIC25_VLLM_PROFILE_ENV, sort_keys=True)}',
    flush=True,
)
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

# Everything the run produces is written here.
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")

## 2. Install the ARC runtime

Install `arc-agi` from the offline competition wheelhouse (the Kaggle submission environment
has no internet).

In [ ]:
# thui-animfast: resolve the competition mount instead of assuming its layout -- Kaggle serves either
# /kaggle/input/competitions/<comp> or /kaggle/input/<comp>, and which one varies between runs.
_COMP_CANDIDATES = ["/kaggle/input/competitions/arc-prize-2026-arc-agi-3", "/kaggle/input/arc-prize-2026-arc-agi-3"]
_COMP_DIR = next((_p for _p in _COMP_CANDIDATES if os.path.isdir(_p)), None)
assert _COMP_DIR is not None, (
    "thui-animfast: no competition mount found. Tried " + repr(_COMP_CANDIDATES)
    + "; /kaggle/input holds "
    + repr(sorted(os.listdir("/kaggle/input")) if os.path.isdir("/kaggle/input") else "MISSING")
)
_WHEELS = os.path.join(_COMP_DIR, "arc_agi_3_wheels")
assert os.path.isdir(_WHEELS), "thui-animfast: resolved wheels dir is not a directory: " + _WHEELS
print("thui-animfast: competition mount = " + _COMP_DIR, flush=True)
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        _WHEELS,
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

## 3. Locate the source bundle

Find the uploaded TAAF source dataset by its marker file, and record where Kaggle mounted
every attached input so setup commands and the solver can find them.

In [ ]:
# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES = ["keithtyser/duck-qwen38-nvfp4-mtp-vllm-smoke-v1", "keithtyser/qwen38-flash-next-vllm-nvfp4-runtime-v1", "jakobbrggen/taaf-kaggle-source-anim-20260807-anim"]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


# Locate the source dataset by its marker file rather than a fixed mount path.
def _find_bundle_dir(label: str) -> Path:
    # thui-animfast: TWO attached datasets carry the marker (his June duck bundle and the anim bundle), so
    # "first marker wins" is a coin flip -- pick by the benchmark_label the marker file records.
    found = {}
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        try:
            found[json.loads(marker.read_text())["benchmark_label"]] = marker.parent
        except Exception as exc:
            print(f"thui-animfast: unreadable marker {marker}: {exc!r}", flush=True)
    if label not in found:
        raise RuntimeError(f"TAAF source bundle {label!r} not found under /kaggle/input; markers = {found}")
    return found[label]


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)


BUNDLE_DIR = _find_bundle_dir("duck-harness-kaggle")          # his: serving_setup.py, vllm patches, watchdog, teardown
ANIM_BUNDLE_DIR = _find_bundle_dir("anim-20260807-anim")     # ours: the solver tree + its pickled benchmark / target
assert BUNDLE_DIR != ANIM_BUNDLE_DIR, "thui-animfast: both labels resolved to one directory"
assert (BUNDLE_DIR / "serving_setup.py").is_file(), f"thui-animfast: his bundle has no serving_setup.py: {BUNDLE_DIR}"
assert (ANIM_BUNDLE_DIR / "src" / "ARC3-Inference" / "inference" / "utils" / "animation.py").is_file(), (
    f"thui-animfast: the anim bundle has no animation.py: {ANIM_BUNDLE_DIR}")
print(f"thui-animfast: anim bundle = {ANIM_BUNDLE_DIR}", flush=True)
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

## 4. Import the bundled source and run solver setup

Put the snapshotted repositories on the path (this process and any child processes), then run
the solver's setup commands â€” installing wheels, fetching model weights, and so on.

In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
# thui-animfast: his tree minus the two solver repos (the June duck), plus the anim solver tree. The loop below
# inserts each entry at sys.path[0], so the LAST entries win -- the anim ones; the .pth is written anim-first.
_SOLVER_REPOS = {"ARC3-Inference", "tufa-arc-agi-framework"}
_his_entries = [e for e in _source_path_entries(BUNDLE_DIR) if e.parent.name not in _SOLVER_REPOS and e.name not in _SOLVER_REPOS]
_anim_entries = _source_path_entries(ANIM_BUNDLE_DIR)
assert _anim_entries and all(str(e).startswith(str(ANIM_BUNDLE_DIR)) for e in _anim_entries), _anim_entries
assert not any(("ARC3-Inference" in str(e) or "tufa-arc-agi-framework" in str(e)) for e in _his_entries), _his_entries
source_entries = _his_entries + _anim_entries
print(f"thui-animfast: source roots his={[str(e) for e in _his_entries]} anim={[str(e) for e in _anim_entries]}", flush=True)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in (_anim_entries + _his_entries)))   # anim first for child processes
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

# [calamitychasm] HAIKU SMOKE TEST: no vLLM. The analyzer is Claude Haiku through Anthropic's
# OpenAI-compatible endpoint, under the served model's harness settings (32,768-token context).
from kaggle_secrets import UserSecretsClient
try:
    _ANTHROPIC_KEY = UserSecretsClient().get_secret('ANTHROPIC_API_KEY')
except Exception as _exc:
    raise RuntimeError('Attach the Kaggle secret ANTHROPIC_API_KEY to this notebook '
                       '(Add-ons > Secrets) and run it again.') from _exc
_HAIKU_ENV = {
    "LOCAL_ANALYZER_APP_NAME": "ARC3 Agent Harness",
    "LOCAL_ANALYZER_CONTEXT_WINDOW": "32768",
    "LOCAL_ANALYZER_TOOL_OUTPUT_TOKENS": "1024",
    "LOCAL_ANALYZER_TOOL_STEPS": "0",
    "LOCAL_ANALYZER_TOOL_TIMEOUT": "30",
    "LOCAL_ANALYZER_TEMPERATURE": "0.6",
    "LOCAL_ANALYZER_TOP_P": "0.95",
    "LOCAL_ANALYZER_TOP_K": "20",
    "LOCAL_ANALYZER_SEED": "20260825",
    "LOCAL_ANALYZER_YIELD_SECONDS": "180",
    "MULTIMODAL_CONTEXT": "current_grid",
    "MULTIMODAL_UPSCALE": "4",
    "LOCAL_ANALYZER_PROVIDER": "anthropic",
    "LOCAL_ANALYZER_BASE_URL": "https://api.anthropic.com/v1",
    "LOCAL_ANALYZER_MODEL_ID": "claude-haiku-4-5",
    "LOCAL_ANALYZER_MAX_OUTPUT": "4096",
    "LOCAL_ANALYZER_ENABLE_THINKING": "false"
}
_persisted = json.loads(SETUP_ENV_PATH.read_text())
_persisted.update(_HAIKU_ENV)                    # the key is NOT written to disk
SETUP_ENV_PATH.write_text(json.dumps(_persisted, indent=2, sort_keys=True) + '\n')
os.environ.update(_HAIKU_ENV)
os.environ['LOCAL_ANALYZER_API_KEY'] = _ANTHROPIC_KEY
assert 'inference' not in sys.modules and 'taaf' not in sys.modules, 'solver imported before env was set'
import inference.agent.tool_agent as _tool_agent
import taaf as _taaf
for _m in (_tool_agent, _taaf):
    assert str(Path(_m.__file__).resolve()).startswith(str(ANIM_BUNDLE_DIR.resolve())), (_m.__name__, _m.__file__)
assert _tool_agent._LOCAL_ANALYZER_CONTEXT_WINDOW == 32768
# Anthropic rejects temperature and top_p together on Claude 4.x models: keep temperature.
import requests as _rq
_orig_post = _rq.post
HAIKU_CALLS = {'n': 0, 'errors': 0}
def _anthropic_post(url, *a, **kw):
    if 'api.anthropic.com' in str(url) and isinstance(kw.get('json'), dict):
        _p = dict(kw['json'])
        if 'temperature' in _p:
            _p.pop('top_p', None)
        kw['json'] = _p
        HAIKU_CALLS['n'] += 1
        _r = _orig_post(url, *a, **kw)
        if getattr(_r, 'status_code', 200) >= 400:
            HAIKU_CALLS['errors'] += 1
            print('HAIKU_HTTP_ERROR', _r.status_code, _r.text[:500], flush=True)
        return _r
    return _orig_post(url, *a, **kw)
_rq.post = _anthropic_post
print('HAIKU_SMOKE analyzer=claude-haiku-4-5 context_window=32768 (served-model setting)', flush=True)


## 5. Load the benchmark

Unpickle the deployment target and the benchmark, stamping the real submission state onto the
target and pointing the benchmark's outputs at the Kaggle working directory.

In [ ]:
# Restore the deployment target and record the real submission state on it.
with open(ANIM_BUNDLE_DIR / "deploy_target.pkl", "rb") as file:   # thui-animfast: the anim bundle's target (32400 s)
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(ANIM_BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:   # thui-animfast: the anim solver
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR
# thui-animfast: the unpickled solver must be the anim chassis, not the June duck.
assert bm.label == "anim-20260807-anim", bm.label
assert getattr(bm.solver, "animation_awareness", None) is True and getattr(bm.solver, "hard_noop_guard", None) is True, vars(bm.solver)
assert type(bm.solver).__module__ == "inference.framework.solver"
print(f"thui-animfast: bm.label={bm.label} solver={type(bm.solver).__name__} animation_awareness={bm.solver.animation_awareness} "
      f"hard_noop_guard={bm.solver.hard_noop_guard} target.max_runtime_s={target.max_runtime_s}", flush=True)


## 6. Customization hook

Optional: tweak `bm`, `bm.games`, or `bm.solver` here before the run starts â€” the safe place
for one-off experiments once the deployed bundle has loaded.

In [ ]:
# [calamitychasm] HAIKU SMOKE settings: 6 games on 3 slots (the queue is exercised), short clocks.
bm.solver.max_runtime_s_per_game = 600.0
bm.solver.analyzer_timeout = 300.0
bm.solver.concurrency = 3
bm.solver.max_actions_per_game = None
bm.solver.save_request_logs = False
bm.solver.start_local_server = False   # no vLLM
bm.solver.model = 'local'              # resolve the analyzer from LOCAL_ANALYZER_* (Haiku)
print('HAIKU_SMOKE settings', bm.solver.max_runtime_s_per_game, bm.solver.concurrency, flush=True)


In [ ]:
# [calamitychasm] MOMENTUM TIME ALLOCATION -- see experiments/stage7_momentum_time.md.
# Replaces _HarnessGameSession.runtime_limit_reached / timing_payload; no bundle file is edited.
import sys as _sys, types as _types
_MT_SRC = r'''"""Momentum time allocation for the anim (Duck/TAAF) solver.

The run is time-bound: every game plays until its fixed 7,920 s clock runs
out. Measured on 225 game-runs from 9 saved public-25 runs of this solver
family (experiments/stage7_momentum_time.md):

  * a turn in a game that leveled up within the last ~9 turns earns 0.423
    completion points; a turn in a game stuck 10+ turns earns 0.155 -- 2.7x
    less -- and stuck turns are 35% of all turns;
  * the level-up rate of a game WITH momentum does not fall with depth
    (0.065 / 0.080 / 0.082 per turn at levels 0-1 / 2-3 / 4+);
  * 48% of games were still progressing when the clock cut them off.

Simulated on a 110-game hidden run (28 slots), merely stopping stuck games
HURTS -- every game gets started anyway, so the freed time has nowhere to go.
What helps is the opposite emphasis: do not cut a game that is still
progressing at 7,920 s, and pay for it by stopping games stuck a long time.

Policy (all times wall-clock seconds, per game, from the moment it starts):
  * stop a game whose last level-up (or start) was >= STALL_S ago;
  * at BASE_S, stop -- unless it leveled up within the last MOMENTUM_S, in
    which case it continues;
  * never beyond HARD_MULT * BASE_S.
The framework's own soft deadline still cancels everything at the notebook
budget, so an extension can never overrun it.

The model is shown `time_remaining_seconds`. Until BASE_S it is shown exactly
what the unmodified solver would show; only an extended game sees the
extension. Request timeouts derive from the same number.

Installed by replacing two methods on `_HarnessGameSession` -- no bundle file
is edited.
"""

from __future__ import annotations

import threading
import time
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

BASE_S = 7920.0
MOMENTUM_S = 45 * 60.0
STALL_S = 100 * 60.0
HARD_MULT = 1.5


@dataclass
class _GameClock:
    started_at: float
    levels: int
    last_progress_at: float
    extended: bool = False
    stopped_reason: Optional[str] = None


@dataclass
class MomentumTimePolicy:
    base_s: float = BASE_S
    momentum_s: float = MOMENTUM_S
    stall_s: float = STALL_S
    hard_mult: float = HARD_MULT
    clock: Callable[[], float] = time.monotonic
    _games: dict = field(default_factory=dict)
    _lock: threading.Lock = field(default_factory=threading.Lock)
    #: One record per game whose clock was changed from the baseline's:
    #: (game_id, reason, elapsed_s, levels). Reasons: "stall", "base_cap",
    #: "extended", "hard_cap".
    decisions: list = field(default_factory=list)

    @property
    def hard_s(self) -> float:
        return self.hard_mult * self.base_s

    # -- bookkeeping -----------------------------------------------------

    def _state(self, session: Any) -> _GameClock:
        key = id(session)
        with self._lock:
            st = self._games.get(key)
            if st is None:
                started = float(getattr(session, "started_at", self.clock()))
                st = _GameClock(started, _levels(session), started)
                self._games[key] = st
            return st

    def _observe(self, session: Any, now: float) -> _GameClock:
        st = self._state(session)
        levels = _levels(session)
        if levels > st.levels:
            st.levels = levels
            st.last_progress_at = now
        return st

    def _record(self, session: Any, st: _GameClock, reason: str, now: float) -> None:
        with self._lock:
            self.decisions.append((_game_id(session), reason, round(now - st.started_at, 1), st.levels))

    # -- the two replaced methods ----------------------------------------

    def limit_reached(self, session: Any) -> bool:
        now = self.clock()
        st = self._observe(session, now)
        if st.stopped_reason is not None:
            return True
        elapsed = now - st.started_at
        since_progress = now - st.last_progress_at
        reason = None
        if since_progress >= self.stall_s:
            reason = "stall"
        elif elapsed >= self.hard_s:
            reason = "hard_cap"
        elif elapsed >= self.base_s:
            if since_progress < self.momentum_s:
                if not st.extended:
                    st.extended = True
                    self._record(session, st, "extended", now)
                return False
            reason = "base_cap"
        if reason is None:
            return False
        st.stopped_reason = reason
        # base_cap at BASE_S on a non-extended game is exactly the baseline's
        # own stop; recorded anyway so the run reports every decision.
        self._record(session, st, reason, now)
        return True

    def effective_limit_s(self, session: Any) -> float:
        """The horizon the game is told about: the baseline's BASE_S, or the
        hard cap once the game has been extended past it."""
        st = self._state(session)
        return self.hard_s if st.extended else self.base_s

    def timing_payload(self, session: Any) -> dict:
        elapsed = max(0.0, self.clock() - self._state(session).started_at)
        remaining = max(0.0, self.effective_limit_s(session) - elapsed)
        return {"run_elapsed_seconds": elapsed, "time_remaining_seconds": remaining}

    # -- reporting ---------------------------------------------------------

    def summary(self) -> dict:
        with self._lock:
            counts: dict = {}
            for _gid, reason, _el, _lv in self.decisions:
                counts[reason] = counts.get(reason, 0) + 1
            return {
                "params": {"base_s": self.base_s, "momentum_s": self.momentum_s,
                           "stall_s": self.stall_s, "hard_mult": self.hard_mult},
                "games_seen": len(self._games),
                "decisions": counts,
                "log": list(self.decisions),
            }


def _levels(session: Any) -> int:
    try:
        return int(session.game.current_state.levels_completed)
    except Exception:  # noqa: BLE001 -- never let bookkeeping stop a game
        return 0


def _game_id(session: Any) -> str:
    run = getattr(getattr(session, "game", None), "game_run", None)
    return str(getattr(run, "game_id", "?"))


def install(session_cls: Any, policy: MomentumTimePolicy) -> None:
    """Replace `runtime_limit_reached` and `timing_payload` on the solver's
    session class. `request_timeout_seconds` reads `timing_payload`, so it
    follows automatically."""
    session_cls.runtime_limit_reached = lambda self: policy.limit_reached(self)
    session_cls.timing_payload = lambda self: policy.timing_payload(self)
    session_cls._momentum_policy = policy
'''
_mt = _types.ModuleType('momentum_time')
_sys.modules['momentum_time'] = _mt   # dataclasses resolve annotations through sys.modules
exec(compile(_MT_SRC, 'momentum_time.py', 'exec'), _mt.__dict__)
import inference.framework.solver as _solver_mod
MOMENTUM_POLICY = _mt.MomentumTimePolicy(base_s=600.0, momentum_s=240.0, stall_s=420.0, hard_mult=1.5)
_mt.install(_solver_mod._HarnessGameSession, MOMENTUM_POLICY)
assert _solver_mod._HarnessGameSession._momentum_policy is MOMENTUM_POLICY
print('MOMENTUM_TIME installed', MOMENTUM_POLICY.summary()['params'], flush=True)


## 7. Run the benchmark

In a real competition rerun (`KAGGLE_IS_COMPETITION_RERUN`), wait for the Kaggle gateway and
play the **live competition Arcade**. Otherwise â€” an interactive "Save & Run" â€” play the
competition's **bundled environment files offline**, with no gateway required, so the notebook
runs end-to-end without a submission. Teardown commands run afterward even if the run raises.

In [ ]:
# Build the live competition game list from the gateway's available environments.
def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# Build the offline game list from the competition's bundled environment files.
def _offline_games(env_dir: str):
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    arcade = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError(f"No offline environments found under {env_dir}.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# The gateway can take a while to come up; poll until it answers.
def _wait_for_gateway(base_url: str, timeout_s: float = 600.0) -> None:
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")


# Skip pre-run display and git-status copies; the staged identity pins the run.

# arc_agi reads RECORDINGS_DIR and ARC_API_KEY from env (ArcadeSpec carries neither); operation
# mode, environments dir, and base url are all passed explicitly via the spec, so no env is needed.
os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

PUBLIC_GAME_IDS = tuple([
    "tn36-ef4dde99",
    "lf52-271a04aa",
    "cn04-2fe56bfb",
    "bp35-0a0ad940",
    "wa30-ee6fef47",
    "lp85-305b61c3",
    "r11l-495a7899",
    "tu93-0768757b",
    "sp80-589a99af",
    "m0r0-492f87ba",
    "vc33-5430563c",
    "ar25-0c556536",
    "ka59-38d34dbb",
    "sc25-635fd71a",
    "sk48-d8078629",
    "dc22-fdcac232",
    "cd82-fb555c5d",
    "ft09-0d8bbf25",
    "g50t-5849a774",
    "ls20-9607627b",
    "re86-8af5384d",
    "s5i5-18d95033",
    "sb26-7fbdac44",
    "su15-1944f8ab",
    "tr87-cd924810"
])

if TRUE_SUBMISSION:
    # Real submission: play the live competition Arcade served by the Kaggle gateway.
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    # The gateway boots asynchronously; wait before swapping in its game list.
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    bm.games = _competition_games()
else:
    # Interactive run: play the bundled competition environments offline (no gateway).
    # The competition's environment files ship alongside the wheelhouse in the competition dataset.
    competition_env_files = str(Path(_COMP_DIR) / "environment_files")   # thui-animfast: resolved in cell 5
    offline_games = _offline_games(competition_env_files)
    offline_by_id = {game.env_name: game for game in offline_games}
    if len(offline_by_id) != len(offline_games):
        raise RuntimeError('The offline public game list contains duplicate IDs.')
    missing = sorted(set(PUBLIC_GAME_IDS) - set(offline_by_id))
    extra = sorted(set(offline_by_id) - set(PUBLIC_GAME_IDS))
    if missing or extra:
        raise RuntimeError(
            f'Offline public game set changed; missing={missing}, extra={extra}.'
        )
    bm.games = [offline_by_id[game_id] for game_id in ['ft09-0d8bbf25', 'vc33-5430563c', 'tu93-0768757b', 'dc22-fdcac232', 'sc25-635fd71a', 'sk48-d8078629']]
    assert len(bm.games) == 6, len(bm.games)
    print(f'HAIKU_SMOKE_SELECTION games={[g.env_name for g in bm.games]}', flush=True)

bm.n_passes = 1
bm.game_weights = None

# Outside a real submission, stop ~10 min before the wall-clock budget for a graceful exit.
budget = float(getattr(target, "max_runtime_s", 0.0) or 0.0)
if budget <= 600.0:
    raise RuntimeError(f'Notebook budget is too small for the teardown reserve: {budget}.')
soft_end = datetime.fromtimestamp(NOTEBOOK_START_EPOCH) + timedelta(
    seconds=budget - 600.0
)

# Start recovery only after setup readiness and all run gates pass.
if str(BUNDLE_DIR) not in sys.path:
    sys.path.insert(0, str(BUNDLE_DIR))
class _NoWatchdog:   # [calamitychasm] no vLLM to watch in the Haiku smoke test
    @staticmethod
    def stop_background(timeout_seconds=0.0):
        return None
vllm_watchdog = _NoWatchdog()

# Play the benchmark; watchdog stop and teardown run even if it raises.
try:
    await bm.run(soft_end_time=soft_end, runtime_environment=target, minimal_diagnostics=True)
    try:   # [calamitychasm] momentum-time decisions, for the write-up
        (WORKING_DIR / 'momentum_time_summary.json').write_text(json.dumps(MOMENTUM_POLICY.summary(), indent=2))
        print('MOMENTUM_TIME decisions', json.dumps(MOMENTUM_POLICY.summary()['decisions']), flush=True)
    except Exception as _exc:
        print('MOMENTUM_TIME summary failed', repr(_exc), flush=True)
    bm._save_json()
    if not TRUE_SUBMISSION:
        # Kaggle Save & Run expects this valid placeholder after an offline run.
        # A real competition rerun uses the live gateway and never enters this branch.
        import pandas as pd

        pd.DataFrame(
            [["1_0", "1", True, 1]],
            columns=["row_id", "game_id", "end_of_game", "score"],
        ).to_parquet(WORKING_DIR / "submission.parquet", index=False)

        # Check terminal coverage, then call the frozen scorer once.
        # This path does not render HTML.
        # [calamitychasm] smoke audit: every game finalized, no crash, policy consulted.
        _runs = list(bm.game_runs)
        _rows = [(r.game_id, r.state, r.levels_completed, len(r.history), r.final_score) for r in _runs]
        for _row in _rows:
            print('HAIKU_SMOKE_RUN', _row, flush=True)
        _summary = MOMENTUM_POLICY.summary()
        _report = {'runs': _rows, 'momentum': _summary, 'haiku_calls': HAIKU_CALLS}
        (WORKING_DIR / 'haiku_smoke_report.json').write_text(json.dumps(_report, indent=2, default=str))
        _problems = []
        if len(_runs) != len(bm.games): _problems.append(f'runs {len(_runs)} != games {len(bm.games)}')
        if any(r.state == 'crashed' for r in _runs): _problems.append('crashed runs')
        if any(r.state not in {'won', 'gave_up', 'cancelled'} or r.final_score is None for r in _runs):
            _problems.append('unfinalized runs')
        if sum(len(r.history) for r in _runs) <= 0: _problems.append('no actions')
        if _summary['games_seen'] != len(_runs): _problems.append('policy did not see every game')
        if HAIKU_CALLS['n'] == 0: _problems.append('no Haiku calls')
        print('HAIKU_SMOKE_REPORT', json.dumps({'decisions': _summary['decisions'], 'haiku_calls': HAIKU_CALLS, 'problems': _problems}), flush=True)
        if _problems:
            raise RuntimeError(f'HAIKU SMOKE FAILED: {_problems}')
        print('HAIKU_SMOKE_PASSED', flush=True)
finally:
    try:
        vllm_watchdog.stop_background(timeout_seconds=15.0)
    finally:
        for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
            print(f"taaf.kaggle: teardown command: {command}", flush=True)
            subprocess.run(
                command,
                shell=True,
                check=False,
                cwd=WORKING_DIR,
                env=_command_env(),
                timeout=30.0,
            )


## 8. Show the diagnostics

A non-submission run writes `diagnostics.html` to `/kaggle/working`; it is rendered inline below
(and downloadable from the working directory). You should be able to click around through the links.

In [ ]:
# Minimal diagnostics are enabled; skip post-run HTML rendering.
